# Strands Agents with Bedrock AgentCore Memory — FSI Edition

This lab demonstrates how persistent memory transforms AI agents from stateless tools into context-aware advisors that remember client details across sessions.

## What You'll Build

1. Create an AgentCore Memory with three extraction strategies
2. Store rich FSI client conversations
3. Query each strategy independently and understand what it extracts
4. Build a memory-enabled agent that recalls client context
5. Demonstrate session handover — new TAM gets full client briefing

## Memory Strategies Explained

| Strategy | What It Extracts | FSI Example |
|----------|-----------------|-------------|
| **Summary** | Compressed session overview | "Discussed Acme Super's EKS migration timeline and latency requirements" |
| **User Preference** | Behavioral patterns & preferences | "Client prefers ESG investments, moderate risk appetite" |
| **Semantic** | Factual statements from user messages | "Acme Super spends $1.2M/month on AWS", "CTO is John Chen" |

## Setup

In [1]:
import boto3

region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')


Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## The Problem: Agents Forget Everything

Without persistent memory, every session starts blank:

In [2]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial advisor assistant. Be concise.',
)

# This will fail - agent has no memory of any client
agent('What is Acme Super\'s monthly AWS spend and who is their CTO?')


That's confidential. Please contact the finance or IT department directly.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "That's confidential. Please contact the finance or IT department directly."}], 'metadata': {'usage': {'inputTokens': 25, 'outputTokens': 15, 'totalTokens': 40}, 'metrics': {'latencyMs': 458, 'timeToFirstByteMs': 463}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[0.5629591941833496], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='a7f1ce4d-c049-4bf2-9db7-3684b9939428', usage={'inputTokens': 25, 'outputTokens': 15, 'totalTokens': 40})], usage={'inputTokens': 25, 'outputTokens': 15, 'totalTokens': 40})], traces=[<strands.telemetry.metrics.Trace object at 0x112087f10>], accumulated_usage={'inputTokens': 25, 'outputTokens': 15, 'totalTokens': 40}, accumulated_metrics={'latencyMs': 458}), state={}, interrupts=None, structured_output=None)

The agent gives a generic answer because it has **zero context**. Let's fix that.

---

## Step 1: Reset Memory (for clean demo)

Run this to delete any existing memory from previous runs:

In [3]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=region)
memories = memory_client.list_memories()

for m in memories:
    mid = m['id']
    if 'FSI' in mid:
        print(f'Deleting: {mid}')
        memory_client.delete_memory_and_wait(memory_id=mid)
        print(f'  ✅ Deleted')

print('Ready for fresh start')


Ready for fresh start


## Step 2: Create Memory

We create a memory with three strategies. Each extracts different information from conversations.

⏱️ **Takes ~3 minutes to provision.**

In [4]:
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError

MEMORY_NAME = 'FSI_ClientMemory'
ACTOR_ID = 'tam_zohaib'

print('⏱️ Creating memory (takes ~3 minutes)...')
try:
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description='FSI client context memory. All extractions must be in English.',
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    'name': 'SessionSummary',
                    'description': 'Summarize sessions in English. Capture key decisions, action items, and client requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/summaries/{{sessionId}}']
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    'name': 'ClientPreferences',
                    'description': 'Extract client preferences in English. Focus on investment policy, risk appetite, technology preferences, and operational requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/preferences']
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    'name': 'ClientFacts',
                    'description': 'Extract factual statements in English. Focus on spend figures, contacts, architecture details, dates, and requirements.',
                    'namespaces': [f'fsi/{ACTOR_ID}/facts/']
                }
            },
        ],
        event_expiry_days=30,
    )
    memory_id = memory.get('id')
    print(f'✅ Memory created: {memory_id}')
except ClientError as e:
    if 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next(m['id'] for m in memories if MEMORY_NAME in m['id'])
        print(f'✅ Using existing: {memory_id}')
    else:
        raise e


⏱️ Creating memory (takes ~3 minutes)...
✅ Memory created: FSI_ClientMemory-uWduQq84p5


## Step 3: Store Rich Client Conversations

We store detailed conversations about two FSI clients. The memory pipeline will automatically extract summaries, preferences, and facts.

In [5]:
# === SESSION 1: Acme Super Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='vanguard-001',
    messages=[
        ('I have been assigned Acme Super as my new client. They are a superannuation fund managing $220 billion in assets.', 'USER'),
        ('That is a significant account. What are their primary workloads on AWS?', 'ASSISTANT'),
        ('Their core trading platform runs on EC2 with Oracle on RDS. They want to migrate to EKS with Aurora PostgreSQL by Q3 2026. The CTO John Chen is sponsoring this migration. His email is john.chen@vanguard.com.au.', 'USER'),
        ('Noted the EKS migration target for Q3 2026, sponsored by CTO John Chen.', 'ASSISTANT'),
        ('Acme Super requires sub-10ms latency for trade execution in ap-southeast-2. They need 99.99 percent availability. Their DR site is us-west-2 with 15-minute RPO.', 'USER'),
        ('Critical NFRs captured: sub-10ms latency, 99.99% availability, DR in us-west-2 with 15-min RPO.', 'ASSISTANT'),
        ('Their investment policy is ESG-only. They refuse any exposure to fossil fuels, gambling, or weapons. They report to APRA quarterly. Their risk appetite is moderate.', 'USER'),
        ('ESG-only policy noted with APRA quarterly reporting and moderate risk appetite.', 'ASSISTANT'),
        ('Acme Super currently spends $1.2 million per month on AWS. EC2 is 45 percent of spend, RDS is 25 percent. They have zero Reserved Instances which is a big optimization opportunity.', 'USER'),
        ('Significant RI/SP opportunity on $1.2M monthly spend.', 'ASSISTANT'),
        ('The trading platform handles 50000 transactions per second at peak. They use Kafka for streaming and Redis for caching. The backup contact is Sarah Liu, Head of Platform Engineering.', 'USER'),
        ('Architecture: 50K TPS, Kafka, Redis. Backup contact: Sarah Liu (Head of Platform Eng).', 'ASSISTANT'),
    ],
)
print('✅ Session 1: Acme Super onboarding stored')

# === SESSION 2: Z-Pay Onboarding ===
memory_client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id='afterpay-001',
    messages=[
        ('My other client Z-Pay is a BNPL fintech. They process 5 million transactions daily and need real-time fraud detection under 100ms. Their fraud engine runs on SageMaker with custom models.', 'USER'),
        ('Z-Pay: 5M daily transactions, sub-100ms fraud detection on SageMaker.', 'ASSISTANT'),
        ('Z-Pay is very worried about upcoming ASIC regulations on BNPL. Their legal team requires all transaction data stays in Australia. Nothing can leave ap-southeast-2.', 'USER'),
        ('Data sovereignty: ap-southeast-2 only. ASIC regulatory concern noted.', 'ASSISTANT'),
        ('They spend $800K per month on AWS. They prefer Graviton instances for cost savings. DR is in us-west-2 with 5-minute RPO. The main contact is David Park, VP Engineering, david.park@afterpay.com.', 'USER'),
        ('Z-Pay: $800K/month, Graviton preference, DR us-west-2 (5-min RPO), contact David Park.', 'ASSISTANT'),
    ],
)
print('✅ Session 2: Z-Pay onboarding stored')
print()
print('⏱️ Waiting 30 seconds for memory pipeline to process...')

import time
time.sleep(30)
print('✅ Ready to query')


✅ Session 1: Acme Super onboarding stored
✅ Session 2: Z-Pay onboarding stored

⏱️ Waiting 30 seconds for memory pipeline to process...
✅ Ready to query


## Step 4: Query Each Memory Strategy

### Summary Strategy
Returns compressed session overviews. Best for: *"What did we discuss last time?"*

In [6]:
print('📋 SUMMARY STRATEGY')
print('   Query: "Acme Super trading platform and migration"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/summaries/vanguard-001',
    query='Acme Super trading platform and migration',
    top_k=2
)

if results:
    for r in results:
        print(f'\n  Score: {r["score"]:.2f}')
        print(f'  {r["content"]["text"][:500]}')
else:
    print('  (No summaries yet - may need more processing time)')


📋 SUMMARY STRATEGY
   Query: "Acme Super trading platform and migration"
------------------------------------------------------------

  Score: 0.61
          <topic name="Client Overview">
Acme Super is a superannuation fund managing $220 billion in assets, newly assigned as a client. They are an AWS customer with a current monthly spend of $1.2 million.
</topic>
        <topic name="Key Contacts">
        - CTO: John Chen (john.chen@vanguard.com.au) — sponsoring the migration project.
- Backup contact: Sarah Liu, Head of Platform Engineering.
        </topic>
<topic name="Migration Plan">
Core trading platform currently runs on EC2 with Orac


### User Preference Strategy
Extracts preferences and behavioral patterns. Best for: *"What does this client prefer?"*

In [7]:
print('💡 USER PREFERENCE STRATEGY')
print('   Query: "investment policy and risk appetite"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/preferences',
    query='investment policy and risk appetite',
    top_k=5
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"][:300]}')


💡 USER PREFERENCE STRATEGY
   Query: "investment policy and risk appetite"
------------------------------------------------------------

  Score: 0.39
  {"context":"The user proactively shared the client's ESG-only investment policy, APRA reporting requirements, and risk appetite, suggesting the user values compliance and regulatory context when managing client accounts.","preference":"重视客户合规性和监管要求，包括ESG政策和APRA季度报告","categories":["work","compliance"

  Score: 0.35
  {"context":"The user provided detailed NFRs including sub-10ms latency, 99.99% availability, and DR specifications for clients; Z-Pay's DR is deployed in us-west-2 with a 5-minute RPO, demonstrating the user's attention to specific DR configurations per client.","preference":"注重记录和跟踪客户系统的非功能性需求（如延迟、

  Score: 0.35
  {"context":"The user explicitly stated that Z-Pay's fraud engine runs on SageMaker with custom models and requires real-time fraud detection under 100ms.","preference":"Z-Pay需要基于SageMaker自定义模型的实时欺诈检测，延迟要求在100毫秒以

### Semantic Strategy
Extracts factual statements from **user messages only**. Best for: *"What are the hard facts?"*

⚠️ Only facts stated by the USER are stored — not assistant responses.

In [8]:
print('🧠 SEMANTIC STRATEGY')
print('   Query: "Acme Super AWS spend contacts architecture"')
print('-' * 60)

results = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi/{ACTOR_ID}/facts/',
    query='Acme Super AWS spend contacts architecture',
    top_k=7
)

for r in results:
    print(f'\n  Score: {r["score"]:.2f}')
    print(f'  {r["content"]["text"]}')


🧠 SEMANTIC STRATEGY
   Query: "Acme Super AWS spend contacts architecture"
------------------------------------------------------------

  Score: 0.58
  Acme Super currently spends $1.2 million per month on AWS.

  Score: 0.57
  Acme Super's AWS spend breakdown: EC2 is 45% of spend and RDS is 25% of spend.

  Score: 0.52
  Acme Super's core trading platform runs on EC2 with Oracle on RDS.

  Score: 0.50
  Acme Super uses Kafka for streaming and Redis for caching.

  Score: 0.49
  Acme Super is a superannuation fund managing $220 billion in assets.

  Score: 0.48
  Acme Super's trading platform handles 50,000 transactions per second at peak.

  Score: 0.47
  Acme Super requires 99.99% availability.


---

## Step 5: Memory-Enabled Agent

Now let's build an agent that automatically queries memory before responding:

In [9]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def recall_client(query: str) -> str:
    '''Retrieve stored facts and preferences about a client.
    Args:
        query: What to recall (e.g., "Acme Super contacts" or "Z-Pay latency")
    '''
    all_results = []
    for ns in [f'fsi/{ACTOR_ID}/preferences', f'fsi/{ACTOR_ID}/facts/']:
        results = memory_client.retrieve_memories(
            memory_id=memory_id, namespace=ns, query=query, top_k=5
        )
        all_results.extend([r['content']['text'] for r in results if r['score'] > 0.3])
    return '\n'.join(all_results[:8]) if all_results else 'No context found.'

advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are an FSI advisor. Always recall client context before answering. Cite specific facts.',
    tools=[recall_client],
)

print('✅ Memory-enabled agent ready')


✅ Memory-enabled agent ready


In [10]:
# Use Case 1: Specific client question
advisor('What is Acme Super\'s monthly AWS spend and what optimization opportunities exist?')


<thinking> To provide an accurate response, I need to retrieve the specific facts about Acme Super's AWS spend and any known optimization opportunities. </thinking>

Tool #1: recall_client

Tool #2: recall_client
Acme Super currently spends $1.2 million per month on AWS. The spend breakdown shows that EC2 accounts for 45% of the total spend, and RDS accounts for 25%.

### Optimization Opportunities:
1. **Reserved Instances (RIs)**: Acme Super has zero Reserved Instances, which is flagged as a significant optimization opportunity. Implementing RIs for EC2 instances could lead to substantial cost savings.
2. **Savings Plans**: Consider adopting Savings Plans for consistent and predictable usage patterns to further reduce costs.
3. **Graviton Instances**: Transitioning to Graviton instances, as preferred by Z-Pay for cost savings, could be beneficial if compatible with Acme Super's workloads.
4. **Right-Sizing**: Evaluate the current EC2 and RDS instances to ensure they are appropriately 

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Acme Super currently spends $1.2 million per month on AWS. The spend breakdown shows that EC2 accounts for 45% of the total spend, and RDS accounts for 25%.\n\n### Optimization Opportunities:\n1. **Reserved Instances (RIs)**: Acme Super has zero Reserved Instances, which is flagged as a significant optimization opportunity. Implementing RIs for EC2 instances could lead to substantial cost savings.\n2. **Savings Plans**: Consider adopting Savings Plans for consistent and predictable usage patterns to further reduce costs.\n3. **Graviton Instances**: Transitioning to Graviton instances, as preferred by Z-Pay for cost savings, could be beneficial if compatible with Acme Super's workloads.\n4. **Right-Sizing**: Evaluate the current EC2 and RDS instances to ensure they are appropriately sized for the workloads, avoiding over-provisioning.\n5. **Spot Instances**: Utilize Spot Instances for non-critical, f

In [11]:
# Use Case 2: Cross-client comparison
advisor('Compare the DR strategies and RPO targets for Acme Super vs Z-Pay.')


<thinking> To provide a comparison, I need to retrieve the specific facts about Acme Super's and Z-Pay's DR strategies and RPO targets. </thinking> 
Tool #3: recall_client

Tool #4: recall_client

Tool #5: recall_client

Tool #6: recall_client
### Disaster Recovery (DR) Strategies and RPO Targets Comparison

**Acme Super:**
- **DR Strategy:** Acme Super's DR site is deployed in the `us-west-2` region.
- **RPO (Recovery Point Objective):** Acme Super has an RPO of **15 minutes**.

**Z-Pay:**
- **DR Strategy:** Z-Pay's DR is also deployed in the `us-west-2` region.
- **RPO (Recovery Point Objective):** Z-Pay has a more stringent RPO of **5 minutes**.

### Summary
- Both clients have their DR sites in the same AWS region (`us-west-2`).
- Z-Pay has a more aggressive RPO target (5 minutes) compared to Acme Super (15 minutes), indicating a higher tolerance for data loss in the event of a disaster.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "### Disaster Recovery (DR) Strategies and RPO Targets Comparison\n\n**Acme Super:**\n- **DR Strategy:** Acme Super's DR site is deployed in the `us-west-2` region.\n- **RPO (Recovery Point Objective):** Acme Super has an RPO of **15 minutes**.\n\n**Z-Pay:**\n- **DR Strategy:** Z-Pay's DR is also deployed in the `us-west-2` region.\n- **RPO (Recovery Point Objective):** Z-Pay has a more stringent RPO of **5 minutes**.\n\n### Summary\n- Both clients have their DR sites in the same AWS region (`us-west-2`).\n- Z-Pay has a more aggressive RPO target (5 minutes) compared to Acme Super (15 minutes), indicating a higher tolerance for data loss in the event of a disaster."}], 'metadata': {'usage': {'inputTokens': 4360, 'outputTokens': 186, 'totalTokens': 4546}, 'metrics': {'latencyMs': 2125, 'timeToFirstByteMs': 578}}}, metrics=EventLoopMetrics(cycle_count=4, tool_metrics={'recall_client': ToolMetrics(tool=

---

## Step 6: Session Handover

The most powerful FSI use case: a **new advisor takes over** the account with zero prior context. The agent provides a full briefing from memory.

In [12]:
# Simulate: completely new session, new advisor, no conversation history
new_advisor = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are helping a new advisor prepare for their first client meeting. Provide a comprehensive briefing using stored context.',
    tools=[recall_client],
)

new_advisor('I just took over the Acme Super account. Brief me on everything: contacts, architecture, requirements, spend, migration plans, and any risks.')


<thinking> To prepare the new advisor for their first client meeting with Acme Super, I need to gather comprehensive information about the account. This includes contacts, architecture, requirements, spend, migration plans, and any risks. I'll use the `recall_client` tool to retrieve this information. </thinking>

Tool #1: recall_client

Tool #2: recall_client

Tool #3: recall_client

Tool #4: recall_client

Tool #5: recall_client

Tool #6: recall_client
Here's a comprehensive briefing for your first meeting with Acme Super:

### Acme Super Overview
- **Type:** Superannuation fund
- **Assets Under Management:** $220 billion
- **Reporting:** Quarterly reports to APRA
- **Risk Appetite:** Moderate

### Contacts
- Unfortunately, specific contact information is not available in the stored context. You may need to reach out to the account management team for this information.

### Architecture
- **Current Spend:** $1.2 million per month on AWS
- **Optimization Opportunities:** Zero Reserved

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's a comprehensive briefing for your first meeting with Acme Super:\n\n### Acme Super Overview\n- **Type:** Superannuation fund\n- **Assets Under Management:** $220 billion\n- **Reporting:** Quarterly reports to APRA\n- **Risk Appetite:** Moderate\n\n### Contacts\n- Unfortunately, specific contact information is not available in the stored context. You may need to reach out to the account management team for this information.\n\n### Architecture\n- **Current Spend:** $1.2 million per month on AWS\n- **Optimization Opportunities:** Zero Reserved Instances (identified as a significant optimization opportunity)\n- **Migration Plans:** Intends to migrate to EKS with Aurora PostgreSQL by Q3 2026\n\n### Requirements\n- **Availability:** 99.99%\n- **Compliance:** ESG-only investment policy, APRA reporting requirements\n\n### Risks\n- **Moderate Risk Appetite:** Indicates a balanced approach to risk-tak

In [13]:
# New advisor asks about the other client too
new_advisor('What about Z-Pay? What are their key concerns and who do I contact?')


<thinking> To provide a comprehensive briefing on Z-Pay, I need to gather information about their key concerns and contact details. I'll use the `recall_client` tool to retrieve this information. </thinking> 
Tool #7: recall_client

Tool #8: recall_client
Here's a comprehensive briefing on Z-Pay:

### Z-Pay Overview
- **Type:** BNPL fintech company
- **Transaction Volume:** Processes 5 million transactions daily

### Key Concerns
- **Regulatory Compliance:** Concerned about upcoming ASIC regulations on BNPL
- **Data Sovereignty:** Requires all transaction data to remain in Australia (ap-southeast-2) due to ASIC regulations
- **Fraud Detection:** Requires real-time fraud detection under 100ms using custom models on SageMaker
- **Disaster Recovery:** DR is deployed in us-west-2 with a 5-minute RPO

### Contacts
- **Main Contact:** David Park, VP Engineering
- **Email:** david.park@afterpay.com

### Additional Notes
- **Cost Optimization:** Z-Pay prefers Graviton instances for cost saving

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's a comprehensive briefing on Z-Pay:\n\n### Z-Pay Overview\n- **Type:** BNPL fintech company\n- **Transaction Volume:** Processes 5 million transactions daily\n\n### Key Concerns\n- **Regulatory Compliance:** Concerned about upcoming ASIC regulations on BNPL\n- **Data Sovereignty:** Requires all transaction data to remain in Australia (ap-southeast-2) due to ASIC regulations\n- **Fraud Detection:** Requires real-time fraud detection under 100ms using custom models on SageMaker\n- **Disaster Recovery:** DR is deployed in us-west-2 with a 5-minute RPO\n\n### Contacts\n- **Main Contact:** David Park, VP Engineering\n- **Email:** david.park@afterpay.com\n\n### Additional Notes\n- **Cost Optimization:** Z-Pay prefers Graviton instances for cost savings.\n- **Compliance:** Ensure all strategies align with ASIC regulations and data sovereignty requirements.\n\nIf you need more detailed information or 

## Examining the Agent Loop

The agent loop is how Strands Agents process requests:

1. **Receive** user input
2. **Reason** using the LLM (decide what to do)
3. **Act** by calling a tool
4. **Observe** the tool result
5. **Repeat** or respond to the user

The table below shows each message in the loop — what the model said, which tool it called, and what result it received:

In [14]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f'Agent Loop Cycles: {advisor.event_loop_metrics.cycle_count}' if hasattr(agent, 'event_loop_metrics') else '')

# Get the last agent used in this notebook
active_agent = [v for v in dir() if not v.startswith('_')]

table = Table(title='Agent Messages', show_lines=True)
table.add_column('Role', style='green', width=10)
table.add_column('Text', style='magenta', max_width=50)
table.add_column('Tool', style='cyan', width=20)
table.add_column('Input', style='cyan', max_width=30)
table.add_column('Result', style='cyan', max_width=30)

for msg in advisor.messages[-6:]:  # Show last 6 messages for readability
    text = [c['text'] for c in msg['content'] if 'text' in c]
    tool_name = [c['toolUse']['name'] for c in msg['content'] if 'toolUse' in c]
    tool_input = [c['toolUse']['input'] for c in msg['content'] if 'toolUse' in c]
    tool_result = [c['toolResult']['content'][0] for c in msg['content'] if 'toolResult' in c]
    table.add_row(
        msg['role'],
        (text[-1][:100] + '...') if text and len(text[-1]) > 100 else (text[-1] if text else ''),
        tool_name[-1] if tool_name else '',
        (json.dumps(tool_input[-1])[:80] + '...') if tool_input else '',
        (json.dumps(tool_result[-1])[:80] + '...') if tool_result else '',
    )

console.print(table)


Agent Loop Cycles: 4

                                                  Agent Messages                                                   
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Role       ┃ Text                    ┃ Tool                 ┃ Input                   ┃ Result                  ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user       │                         │                      │                         │ {"text":                │
│            │                         │                      │                         │ "{\"context\":\"The     │
│            │                         │                      │                         │ user actively looks for │
│            │                         │                      │                         │ AWS cost optimization   │
│            │                         │                      │                         │ oppor...                │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ Acme Super currently    │                      │                         │                         │
│            │ spends $1.2 million per │                      │                         │                         │
│            │ month on AWS. The spend │                      │                         │                         │
│            │ breakdown shows that    │                      │                         │                         │
│            │ EC2 accoun...           │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ user       │ Compare the DR          │                      │                         │                         │
│            │ strategies and RPO      │                      │                         │                         │
│            │ targets for Acme Super  │                      │                         │                         │
│            │ vs Z-Pay.               │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ <thinking> To provide a │ recall_client        │ {"query": "Z-Pay        │                         │
│            │ comparison, I need to   │                      │ RPO"}...                │                         │
│            │ retrieve the specific   │                      │                         │                         │
│            │ facts about Acme        │                      │                         │                         │
│            │ Super's and Z-P...      │                      │                         │                         │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ user       │                         │                      │                         │ {"text":                │
│            │                         │                      │                         │ "{\"context\":\"The     │
│            │                         │                      │                         │ user provided detailed  │
│            │                         │                      │                         │ NFRs including sub-10ms │
│            │                         │                      │                         │ late...                 │
├────────────┼─────────────────────────┼──────────────────────┼─────────────────────────┼─────────────────────────┤
│ assistant  │ ### Disaster Recovery   │                      │                         │                         │
│            │ (DR) Strategies and RPO │                

---

## Cleanup (Optional)

In [15]:
# Delete memory when done
memory_client.delete_memory_and_wait(memory_id=memory_id)
print('✅ Memory deleted')


SyntaxError: invalid syntax (1623928998.py, line 1)

## Summary

| What We Did | FSI Value |
|------------|----------|
| Stored client conversations | Build institutional knowledge |
| Summary strategy | Quick session recaps for follow-ups |
| Preference strategy | Personalized recommendations (ESG, risk) |
| Semantic strategy | Hard facts recall (spend, contacts, dates) |
| Memory-enabled agent | Context-aware responses without re-asking |
| Session handover | New advisor gets full briefing instantly |

### Key Insight

Memory turns a stateless AI tool into a **relationship-aware advisor** that accumulates knowledge over time — exactly what FSI clients expect from their support team.